### Imports

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
from scrapers import barbora, rimi
from scrapers.common import save_rows

### Scrape with guardrail

In [ ]:
rows_b = barbora.scrape("c/LVOK/mesa-zuvis-ir-kulinarija/sviezia-mesa-ir-paukstiena", max_pages=20)
if rows_b:
    print(save_rows(rows_b, "barbora", "fresh-meat"))
else:
    print("Barbora returned nothing, keeping the old file")

In [ ]:

rows_r = rimi.scrape("SH-9", "mesa-ir-zuvis-", max_pages=20)
if rows_r:
    print(save_rows(rows_r, "rimi", "meat-and-fish"))
else:
    print("Rimi returned nothing, keep the old file")

In [ ]:
import pandas as pd
from scrapers.common import DATA_DIR

def load_store(store):
    files = sorted((DATA_DIR / store).glob("*.json"))
    return pd.concat([pd.read_json(f, dtype={"id": str}).assign(file=f.stem) for f in files])

b, r = load_store("barbora"), load_store("rimi")

for name, df in [("BARBORA", b), ("RIMI", r)]:
    print(f"\n===== {name}: {len(df)} rows =====")

    # 1. duplicates
    print("duplicate ids:", df["id"].duplicated().sum())

    # 2. required fields filled
    for c in ["id", "name", "price", "unit_price", "unit", "url"]:
        print(f"missing {c}:", df[c].isna().sum())

    # 3. sane values
    print("price <= 0:", (df["price"] <= 0).sum(),
          "| unit_price <= 0:", (df["unit_price"] <= 0).sum())
    print("units:", df["unit"].value_counts().to_dict())   # expect mostly kg / l / vnt

    # 4. promo logic
    p = df[df["on_promo"] == True]
    print("promo rows:", len(p), f"({len(p)/len(df):.0%})")
    print("promo with price >= regular_price:", (p["price"] >= p["regular_price"]).sum())

    # 5. coverage: what categories did we actually get?
    print(df["category"].str.split("/").str[:3].str.join("/").value_counts().head(8))

# 6. same columns in both stores
print("\nschema identical:", set(b.columns) == set(r.columns))
print("differences:", set(b.columns) ^ set(r.columns))

In [ ]:
b[b["unit"] != "kg"]